## Fine-tuning Evo2's `<BOS>` embedding to generate E. coli σ70 promoters

### Background and motivation

Evo2 was pretrained on packed documents of the form `|tax|<sequence><EOS>|tax|<sequence><EOS>…` with no `<BOS>` token in the input stream. The HF tokenizer used by this recipe does reserve `<BOS>` at id 2, but its embedding row was never updated by any pretraining gradient — it sits at whatever the embedding initializer produced.

This notebook treats that unused row as a blank slate. We **freeze the entire backbone** and train **only embedding row 2**, conditioning the model to emit a specific class of short sequences when prompted with `<BOS>`. We use E. coli σ70 promoters as the target distribution because they are short (~80 bp), well-characterized, and have crisp diagnostic motifs (`-10` TATAAT and `-35` TTGACA boxes) that make a before-vs-after motif logo plot the punchline of the demo.

If a single trained embedding vector can carry enough signal to produce sequences with the canonical σ70 architecture, that's a striking result — the smallest possible LoRA-style intervention you can make to a frozen genomic foundation model.

### What this notebook does
1. Downloads E. coli σ70 promoters from RegulonDB (GFF3 format).
2. Filters, deduplicates, canonicalizes, and frames each promoter to a uniform 80 bp window centered on the TSS.
3. Packs `<BOS>seq<EOS>` records into FASTA documents, runs `preprocess_evo2` to produce a Megatron-format dataset.
4. Generates baseline samples from the **pretrained** Evo2 with `<BOS>` as the prompt — no learned signal yet.
5. Fine-tunes only embedding row 2 (PEFT `TrainableTokensConfig`, target `embedding.word_embeddings`), masking BOS positions from the loss.
6. Generates samples from the **fine-tuned** model with `<BOS>` as the prompt.
7. Compares before-vs-after via motif logos, `-10`/`-35` motif scanning, and GC distributions.

### Requirements

Same hardware envelope as the parent fine-tuning tutorial (~45 GB GPU memory for Evo2-1B). Promoter sequences are short, so the actual fine-tuning step is much faster than the chr20/21/22 demo — most of the wall-clock time is checkpoint download and conversion.

> **API note.** As of this writing, `train_evo2` does not yet expose a `--peft-config` flag for trainable-tokens-only fine-tuning. The cells below assume that API and produce a YAML config in the form expected by `peft.TrainableTokensConfig`, with `target_modules: ["embedding.word_embeddings"]` matching Megatron's embedding module path. The notebook also assumes a hook that consumes `ignore_token_indices_in_loss` to mask `<BOS>` positions out of the cross-entropy. When the wrapper lands, this notebook should run as-is.

In [ ]:
import os
from pathlib import Path

FAST_CI_MODE: bool = bool(int(os.environ.get("FAST_CI_MODE", "0")))

CLEANUP: bool = False
if CLEANUP:
    !rm -rf bos_promoter_data bos_promoter_preprocessed bos_promoter_run
    !rm -f PromoterSet.gff3 PromoterSet.tsv promoters_raw.tsv promoters_filtered.tsv
    !rm -f bos_preprocess_config.yaml bos_training_data_config.yaml bos_peft_config.yaml
    !rm -f baseline_samples.jsonl finetuned_samples.jsonl baseline_prompts.jsonl finetuned_prompts.jsonl

### Step 1 — Acquire E. coli σ70 promoters from RegulonDB

[RegulonDB](https://regulondb.ccg.unam.mx) is the canonical curated database of E. coli K-12 transcriptional regulation. The `PromoterSet.gff3` file lists ~3,500 promoters annotated by sigma factor and evidence quality, in standard GFF3 format with promoter metadata in the column-9 attributes (`SigmaFactor=`, `Sequence=`, `Confidence=`, …).

If the download URL has changed in a newer release, replace the URL below or place a local copy of `PromoterSet.gff3` in the working directory before running this cell.

In [ ]:
REGULONDB_URL = "https://regulondb.ccg.unam.mx/media/raw/gff3/PromoterSet.gff3"

if not os.path.exists("PromoterSet.gff3"):
    !wget --no-check-certificate -O PromoterSet.gff3 {REGULONDB_URL}

!wc -l PromoterSet.gff3

### Step 2 — Filter, canonicalize, deduplicate, and frame around the TSS

Preprocessing decisions, and why each one matters:

1. **σ70 only.** Other sigma factors (σ24, σ28, σ32, σ38, σ54) bind different consensus motifs. Mixing them muddles the motif-logo signal.
2. **Confirmed / Strong evidence only.** Drops weak / computational predictions — keeps the training signal clean.
3. **Uppercase, ACGT only.** Drop sequences containing IUPAC ambiguity codes.
4. **No reverse-complement augmentation.** Promoters are directional — RC-augmenting would teach the model something biologically wrong.
5. **Frame to a uniform 80 bp window** (-60 to +20 around TSS). Makes the motif logo trivially interpretable: every position has the same biological coordinate.
6. **Deduplicate.** Hash-level dedup removes exact duplicates from redundant annotations. CD-HIT-EST clustering at 90% would be stricter; for this demo the exact-dedup is sufficient.
7. **Train/val/test split AFTER dedup.** Prevents near-identical sequences from leaking across splits.

In [ ]:
import pandas as pd
import re

# RegulonDB ships PromoterSet as GFF3: 9 tab-separated columns where column 9 is a semicolon-separated
# attribute string. We pull SigmaFactor / Sequence / Confidence from the attributes. Attribute keys
# have varied across releases (e.g. "SigmaFactor" vs "Sigma_factor"), so we look up case-insensitive
# with a list of fallback names.

ATTR_FALLBACKS = {
    "sigma": ["sigmafactor", "sigma_factor", "sigma"],
    "sequence": ["sequence", "promoter_sequence", "seq"],
    "evidence": ["confidence", "confidencelevel", "evidence_type", "evidence"],
    "name": ["name", "id", "promoter_name"],
}

def parse_gff3_attributes(attr_str):
    """Parse the 9th GFF3 column 'k1=v1;k2=v2' into a dict (URL-decoded)."""
    from urllib.parse import unquote
    out = {}
    for kv in attr_str.split(";"):
        if "=" in kv:
            k, v = kv.split("=", 1)
            out[k.strip().lower()] = unquote(v.strip())
    return out

def lookup(attrs, candidates):
    for c in candidates:
        if c in attrs and attrs[c]:
            return attrs[c]
    return ""

records = []
with open("PromoterSet.gff3") as f:
    for line in f:
        if line.startswith("#") or not line.strip():
            continue
        parts = line.rstrip("\n").split("\t")
        if len(parts) < 9:
            continue
        attrs = parse_gff3_attributes(parts[8])
        records.append({
            "promoter_name": lookup(attrs, ATTR_FALLBACKS["name"]),
            "strand": parts[6],
            "sigma": lookup(attrs, ATTR_FALLBACKS["sigma"]),
            "sequence": lookup(attrs, ATTR_FALLBACKS["sequence"]),
            "evidence_type": lookup(attrs, ATTR_FALLBACKS["evidence"]),
        })
df = pd.DataFrame(records)
print(f"Loaded {len(df)} GFF3 records")
if len(df) == 0:
    raise RuntimeError("No records parsed — check that PromoterSet.gff3 is the expected GFF3 file, not an HTML error page.")
print("Sample sigma values:", df["sigma"].value_counts().head(8).to_dict())
print("Sample evidence values:", df["evidence_type"].value_counts().head(8).to_dict())

# Filter to σ70 with high-quality evidence.
df = df[df["sigma"].str.contains("Sigma70", case=False, na=False)]
df = df[df["evidence_type"].str.contains("Confirmed|Strong", case=False, na=False)]
print(f"After σ70 + Confirmed/Strong filter: {len(df)}")

# Canonicalize: uppercase, drop anything with non-ACGT bases.
df["sequence"] = df["sequence"].astype(str).str.upper().str.strip()
df = df[df["sequence"].apply(lambda s: bool(s) and bool(re.fullmatch(r"[ACGT]+", s)))]
print(f"After ACGT-only filter: {len(df)}")

# Frame to 80 bp around TSS. RegulonDB sequences are typically -60..+20 (81 bp with TSS marked); take the first 80 bp.
df = df[df["sequence"].str.len() >= 80].copy()
df["sequence"] = df["sequence"].str[:80]
print(f"After 80 bp framing: {len(df)}")

# Hash-level deduplication.
df = df.drop_duplicates("sequence").reset_index(drop=True)
print(f"After deduplication: {len(df)} unique promoters")

# Train / val / test split (post-dedup).
df = df.sample(frac=1.0, random_state=12342).reset_index(drop=True)
n = len(df)
n_train = int(0.8 * n); n_val = int(0.1 * n)
splits = {
    "train": df.iloc[:n_train].copy(),
    "val":   df.iloc[n_train:n_train + n_val].copy(),
    "test":  df.iloc[n_train + n_val:].copy(),
}
for name, sub in splits.items():
    print(f"  {name}: {len(sub)} sequences")

splits["train"].to_csv("promoters_filtered.tsv", sep="\t", index=False)

### Sanity check — does the training set actually contain the σ70 motifs?

If you can't see clear `-10` and `-35` consensus boxes in the training-set position-frequency matrix, the fine-tune has nothing to latch onto. This cell plots the position-wise base frequency for the training split. You should see:
- **Position ~45–50 (relative to start of the 80 bp window):** TATAAT-like enrichment (the `-10` box, ~10 bp upstream of TSS at position 60).
- **Position ~20–26:** TTGACA-like enrichment (the `-35` box).
- Roughly random composition elsewhere.

In [ ]:
# Install plotting dependencies if missing.
try:
    import logomaker  # noqa: F401
except ImportError:
    !pip install -q logomaker

In [ ]:
import numpy as np
import logomaker
import matplotlib.pyplot as plt

BASES = list("ACGT")

def position_frequency_matrix(seqs):
    L = len(seqs[0])
    counts = np.zeros((L, 4))
    for s in seqs:
        for i, b in enumerate(s):
            j = BASES.index(b)
            counts[i, j] += 1
    return pd.DataFrame(counts / counts.sum(axis=1, keepdims=True), columns=BASES)

def plot_logo(seqs, title, ax):
    """Plot an information-content logo. Gracefully handles the empty-input case (e.g. when
    pretrained-baseline `<BOS>` generations produce no valid 80-bp ACGT sequences)."""
    if not seqs:
        ax.text(0.5, 0.5, "no valid samples", ha="center", va="center", transform=ax.transAxes, fontsize=12)
        ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
        return
    pfm = position_frequency_matrix(seqs)
    info = logomaker.transform_matrix(pfm, from_type="probability", to_type="information")
    logomaker.Logo(info, ax=ax, color_scheme="classic")
    ax.set_title(title); ax.set_xlabel("position"); ax.set_ylabel("bits")

fig, ax = plt.subplots(figsize=(14, 2.5))
plot_logo(splits["train"]["sequence"].tolist(), f"Training set (n={len(splits['train'])})", ax)
plt.tight_layout(); plt.show()

# GC content distribution.
def gc(s): return (s.count("G") + s.count("C")) / len(s)
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist([gc(s) for s in splits["train"]["sequence"]], bins=30)
ax.set_xlabel("GC fraction"); ax.set_ylabel("count")
ax.set_title("Training set GC content")
plt.tight_layout(); plt.show()

### Step 3 — Pack `<BOS>seq<EOS>` records into FASTA

Each Evo2 training context is 8,192 tokens. An 80 bp promoter alone barely fills 1% of a window, so we **pack many promoters per FASTA record** with the explicit `<BOS>` and `<EOS>` markers between them:

```
<BOS>GCTTGACA…TATAAT…<EOS><BOS>GCTTGACA…TATAAT…<EOS>…
```

The HF fast tokenizer treats `<BOS>` and `<EOS>` as atomic special tokens (ids 2 and 0 respectively) — its added-tokens matcher fires before the per-character `Split` pre-tokenizer runs. So this string tokenizes to `[2, byte, byte, …, byte, 0, 2, byte, …]`.

Each FASTA record below contains roughly enough `<BOS>seq<EOS>` units to fill an 8k context (~95 promoters × ~85 tokens ≈ 8k). The `preprocess_evo2` step then turns this into a Megatron `IndexedDataset`.

In [ ]:
data_dir = Path("bos_promoter_data"); data_dir.mkdir(exist_ok=True)

PROMOTERS_PER_RECORD = 90  # ~90 * 85 ≈ 7.6k tokens, leaves headroom under 8k.

def write_packed_fasta(seqs, out_path, record_prefix):
    with open(out_path, "w") as f:
        for chunk_idx, start in enumerate(range(0, len(seqs), PROMOTERS_PER_RECORD)):
            chunk = seqs[start:start + PROMOTERS_PER_RECORD]
            packed = "".join(f"<BOS>{s}<EOS>" for s in chunk)
            f.write(f">{record_prefix}_{chunk_idx:04d}\n{packed}\n")

for name, sub in splits.items():
    write_packed_fasta(sub["sequence"].tolist(), data_dir / f"{name}.fa", name)
    print(f"  wrote {name}.fa: {(data_dir / f'{name}.fa').stat().st_size / 1024:.1f} KB")

# Concatenate splits into one FASTA — preprocess_evo2 splits on its own using train_split / valid_split / test_split.
# We override that here by running preprocess once per split file with split=1.0/0/0 etc.
# Simpler: cat all splits and let the tool split. But since we want exact split control, we run it three times.
!ls -lh {data_dir}/

### Step 4 — Tokenize with `preprocess_evo2`

We run the BioNeMo preprocessing pipeline once per split so the train/val/test boundaries we just chose are honored exactly. Each invocation produces a `.bin`/`.idx` pair under `bos_promoter_preprocessed/`.

Important config flags:
- `embed_reverse_complement: false` — promoters are directional, no RC augmentation.
- `random_lineage_dropout: 0.0` — we are NOT injecting taxonomy tokens; the trigger is `<BOS>` alone.
- `append_eod: false` — we already wrote `<EOS>` markers explicitly inside each FASTA record. Letting the preprocessor add another at document end would just give two consecutive EOS bytes (harmless but redundant).

In [ ]:
from bionemo.evo2.data.dataset_tokenizer import DEFAULT_HF_TOKENIZER_MODEL_PATH_512

preproc_dir = Path("bos_promoter_preprocessed"); preproc_dir.mkdir(exist_ok=True)

config_entries = []
for split_name in ["train", "val", "test"]:
    in_fa = (data_dir / f"{split_name}.fa").absolute()
    train_pct = 1.0 if split_name == "train" else 0.0
    val_pct   = 1.0 if split_name == "val"   else 0.0
    test_pct  = 1.0 if split_name == "test"  else 0.0
    config_entries.append(f"""
- datapaths: ["{in_fa}"]
  output_dir: "{preproc_dir.absolute()}"
  output_prefix: bos_promoter_{split_name}
  train_split: {train_pct}
  valid_split: {val_pct}
  test_split: {test_pct}
  overwrite: true
  embed_reverse_complement: false
  random_reverse_complement: 0.0
  random_lineage_dropout: 0.0
  include_sequence_id: false
  transcribe: "back_transcribe"
  force_uppercase: true
  indexed_dataset_dtype: "uint8"
  hf_tokenizer_model_path: {DEFAULT_HF_TOKENIZER_MODEL_PATH_512}
  pretrained_tokenizer_model: null
  special_tokens: null
  fast_hf_tokenizer: true
  append_eod: false
  enforce_sample_length: null
  ftfy: false
  workers: 1
  preproc_concurrency: 100000
  chunksize: 25
  drop_empty_sequences: true
  nnn_filter: false
  seed: 12342
""")

with open("bos_preprocess_config.yaml", "w") as f:
    f.write("".join(config_entries))

!preprocess_evo2 --config bos_preprocess_config.yaml
!ls -lh {preproc_dir}/

### Step 5 — Acquire the base Evo2 1B checkpoint

Same workflow as the parent fine-tuning tutorial: download the NeMo2 checkpoint from NGC and convert to MBridge format. The `evo2_1b_base` checkpoint is the smallest one available and runs comfortably on a single A6000.

In [ ]:
from bionemo.core.data.load import load
from bionemo.core.utils.subprocess_utils import run_subprocess_safely

mbridge_ckpt_path = Path("evo2_1b_bf16_mbridge")
if not mbridge_ckpt_path.exists():
    nemo2_ckpt_path = load("evo2/1b-8k-bf16:1.0")
    convert_cmd = f"""evo2_convert_nemo2_to_mbridge \
        --nemo2-ckpt-dir {nemo2_ckpt_path} \
        --mbridge-ckpt-dir {mbridge_ckpt_path} \
        --model-size evo2_1b_base \
        --mixed-precision-recipe bf16_mixed \
        --seq-length 8192 \
        --tokenizer-path {DEFAULT_HF_TOKENIZER_MODEL_PATH_512}"""
    print(f"Running: {convert_cmd}")
    result = run_subprocess_safely(convert_cmd)

print(f"MBridge checkpoint: {mbridge_ckpt_path.absolute()}")

### Step 6 — Baseline: generate from the **pretrained** model with `<BOS>`

Before fine-tuning, we sample from the pretrained checkpoint with `<BOS>` as the prompt. The BOS embedding row was never updated during pretraining, so this should produce essentially garbage / noise — and *certainly* no σ70 motif structure. This is the "before" half of the punchline.

In [ ]:
import json

N_BASELINE_SAMPLES = 16 if FAST_CI_MODE else 200

# Build a JSONL of identical <BOS> prompts so infer_evo2 produces independent samples in batch.
with open("baseline_prompts.jsonl", "w") as f:
    for i in range(N_BASELINE_SAMPLES):
        f.write(json.dumps({"id": f"baseline_{i:04d}", "prompt": "<BOS>"}) + "\n")

baseline_cmd = f"""infer_evo2 \
    --ckpt-dir {mbridge_ckpt_path} \
    --prompt-file baseline_prompts.jsonl \
    --max-new-tokens 80 \
    --temperature 1.0 \
    --top-p 1.0 \
    --max-batch-size 16 \
    --output-file baseline_samples.jsonl"""

print(f"Running: {baseline_cmd}")
result = run_subprocess_safely(baseline_cmd)
if result["returncode"] != 0:
    print("STDOUT:", result["stdout"][-2000:])
    print("STDERR:", result["stderr"][-2000:])
    raise RuntimeError("Baseline generation failed")

!head -3 baseline_samples.jsonl

### Step 7 — Fine-tune **only** the `<BOS>` embedding row

We freeze the entire Evo2 backbone and train **only embedding row 2** (the `<BOS>` token) via PEFT's `trainable_token_indices`. Two design choices worth flagging:

**Loss masking on `<BOS>` positions.** The job of the BOS row is to be a useful *input embedding* — when it's present in the context, it should steer prediction of the next 80 bp toward promoter structure. The same row also serves as the *output logit* for predicting BOS (HF/Megatron embeddings are tied by default). If we leave BOS in the labels, the model is also trained to *emit* BOS after a previous EOS. With tied weights, that pulls the same row in two directions: be a good input conditioning vector AND be a good output logit. So we set the label to the ignore index (`-100`) at every position whose target is `<BOS>`. EOS stays in the labels — we want the model to learn when to stop.

**Pure embedding-only, no LoRA on attention.** This is the strict-minimum baseline. If a single trained vector is too weak, the natural extension is to add LoRA on a few early attention blocks — but that would change the headline result from "one trained vector" to "one vector plus low-rank attention adapters," which is a different (still interesting) claim.

> **API note (assumed).** The `peft_config.yaml` and `--peft-config` flag below describe the API we plan to support but is not yet wired through `train_evo2`. The structure mirrors how PEFT and BioNeMo already cooperate on the inference path (`run_config["peft"]` in `infer.py`). When the wrapper lands, this cell should run as-is.

In [ ]:
import torch

BOS_TOKEN_ID = 2
EOS_TOKEN_ID = 0

# PEFT config: train only the embedding row for <BOS> (id 2). Megatron names the input embedding
# `embedding.word_embeddings` (HF-style `embed_tokens` would not match this model). We use the
# dedicated `TrainableTokensConfig` rather than `LoraConfig(target_modules=[])` so the intent
# ("only this row, nothing else") is encoded structurally.
#
# ignore_token_indices_in_loss is consumed by the (assumed) train_evo2 loss-mask hook to drop
# BOS positions from the cross-entropy. With tied embeddings, leaving BOS in the labels would
# pull the same row toward being a good *output* logit, conflicting with its job as an *input*
# conditioning vector.
peft_yaml = f"""
_target_: peft.TrainableTokensConfig
target_modules: ["embedding.word_embeddings"]
token_indices: [{BOS_TOKEN_ID}]
ignore_token_indices_in_loss: [{BOS_TOKEN_ID}]
"""
with open("bos_peft_config.yaml", "w") as f:
    f.write(peft_yaml.lstrip())

# Training data config — points at the .bin/.idx outputs from preprocess_evo2.
out_pfx = preproc_dir.absolute() / "bos_promoter"
data_yaml = f"""
- dataset_prefix: {out_pfx}_train_nucleotide_fast_tokenizer_512_train
  dataset_split: train
  dataset_weight: 1.0
- dataset_prefix: {out_pfx}_val_nucleotide_fast_tokenizer_512_val
  dataset_split: validation
  dataset_weight: 1.0
- dataset_prefix: {out_pfx}_test_nucleotide_fast_tokenizer_512_test
  dataset_split: test
  dataset_weight: 1.0
"""
with open("bos_training_data_config.yaml", "w") as f:
    f.write(data_yaml.lstrip())

MAX_STEPS = 20 if FAST_CI_MODE else 300
warmup = min(MAX_STEPS, 50)
num_gpus = torch.cuda.device_count()

train_cmd = f"""torchrun --nproc_per_node={num_gpus} --no-python train_evo2 \
    -d bos_training_data_config.yaml \
    --dataset-dir {preproc_dir} \
    --result-dir bos_promoter_run \
    --experiment-name evo2_bos_promoters \
    --context-parallel-size {num_gpus} \
    --model-size evo2_1b_base \
    --seq-length 8192 \
    --micro-batch-size 1 --global-batch-size 8 \
    --eval-iters 5 --eval-interval {min(MAX_STEPS // 2, 50)} \
    --decay-steps 100000 --warmup-steps {warmup} \
    --hf-tokenizer-model-path {DEFAULT_HF_TOKENIZER_MODEL_PATH_512} \
    --lr 0.001 --min-lr 0.0001 \
    --max-steps {MAX_STEPS} \
    --finetune-ckpt-dir {mbridge_ckpt_path} \
    --peft-config bos_peft_config.yaml \
    --clip-grad 1.0 --wd 0.0 \
    --activation-checkpoint-recompute-num-layers 5"""

print(f"Running: {train_cmd}")
result = run_subprocess_safely(train_cmd)
if result["returncode"] != 0:
    print("STDOUT:", result["stdout"][-2000:])
    print("STDERR:", result["stderr"][-2000:])
    raise RuntimeError("Fine-tuning failed")

### Step 8 — Inspect training metrics

With only one trainable vector and a strong learning rate, you should see `lm loss` drop quickly within the first 50–100 steps and then plateau. If the loss never moves, the most likely culprits are: (a) the BOS row already happens to evoke a benign distribution (unlikely with 1B random init), (b) the trigger token is being masked from inputs as well as labels (it shouldn't be — input mask and loss mask are separate), or (c) tied embeddings are not being updated through the trainable-tokens path.

In [ ]:
import seaborn as sns
import tensorboard.backend.event_processing.event_accumulator as event_accumulator

def tensorboard_to_dataframe(event_file):
    ea = event_accumulator.EventAccumulator(event_file, size_guidance={event_accumulator.SCALARS: 0})
    ea.Reload()
    tags = ea.Tags()["scalars"]
    all_steps = sorted({event.step for tag in tags for event in ea.Scalars(tag)})
    out = pd.DataFrame({"step": all_steps})
    for tag in tags:
        step_to_value = {event.step: event.value for event in ea.Scalars(tag)}
        out[tag] = out["step"].map(step_to_value)
    return out

def plot_metrics(df, metrics):
    n = len(metrics)
    fig, axes = plt.subplots(n, 1, figsize=(15, 3 * n), sharex=True)
    if n == 1: axes = [axes]
    sns.set_style("whitegrid")
    for i, m in enumerate(metrics):
        if m in df.columns:
            sns.lineplot(x="step", y=m, data=df, ax=axes[i], linewidth=2.5)
            axes[i].set_title(m)
    plt.tight_layout(); plt.show()

log_dirs = !find bos_promoter_run -name "events.out.tfevents*" | sort | tail -1
if log_dirs and log_dirs[0]:
    df_tb = tensorboard_to_dataframe(log_dirs[0])
    plot_metrics(df_tb, ["lm loss", "learning-rate", "grad-norm", "lm loss validation"])

### Step 9 — Generate from the **fine-tuned** model with `<BOS>`

Same prompt as the baseline (`<BOS>`), same sampling parameters. Any structure in the outputs is attributable to the single embedding vector we trained.

In [ ]:
N_FINETUNED_SAMPLES = 16 if FAST_CI_MODE else 200

with open("finetuned_prompts.jsonl", "w") as f:
    for i in range(N_FINETUNED_SAMPLES):
        f.write(json.dumps({"id": f"finetuned_{i:04d}", "prompt": "<BOS>"}) + "\n")

ft_ckpt = Path("bos_promoter_run/evo2_bos_promoters/checkpoints")
ft_iter = sorted(ft_ckpt.glob("iter_*"))[-1] if ft_ckpt.exists() else ft_ckpt

ft_cmd = f"""infer_evo2 \
    --ckpt-dir {ft_iter} \
    --prompt-file finetuned_prompts.jsonl \
    --max-new-tokens 80 \
    --temperature 1.0 \
    --top-p 1.0 \
    --max-batch-size 16 \
    --output-file finetuned_samples.jsonl"""

print(f"Running: {ft_cmd}")
result = run_subprocess_safely(ft_cmd)
if result["returncode"] != 0:
    print("STDOUT:", result["stdout"][-2000:])
    print("STDERR:", result["stderr"][-2000:])
    raise RuntimeError("Fine-tuned generation failed")

### Step 10 — Validation: motif logos, motif scanning, GC distributions

Three diagnostics, ordered from most to least visual:

1. **Motif logos.** Position-frequency matrices for the three populations (training set, baseline samples, fine-tuned samples) plotted as information-content logos via `logomaker`. The training-set logo shows the canonical `-10`/`-35` boxes. The baseline logo should be flat. The fine-tuned logo should reproduce the same boxes if the BOS row succeeded.
2. **Motif scanning.** PWM-score each generated sequence against the σ70 `-10` (TATAAT) and `-35` (TTGACA) consensus matrices, count how many sequences hit both motifs above threshold, and at the right spacing.
3. **GC content.** Check that fine-tuned samples have GC distribution close to the training set. Helps catch degenerate generations like "AAAA…" repeats.


In [ ]:
def load_completions(jsonl_path):
    seqs = []
    with open(jsonl_path) as f:
        for line in f:
            obj = json.loads(line)
            s = obj["completion"].upper()
            # Keep only the first 80 ACGT bases (drop any non-ACGT or trailing special tokens).
            clean = re.sub(r"[^ACGT]", "", s)[:80]
            if len(clean) == 80:
                seqs.append(clean)
    return seqs

baseline_seqs  = load_completions("baseline_samples.jsonl")
finetuned_seqs = load_completions("finetuned_samples.jsonl")
training_seqs  = splits["train"]["sequence"].tolist()

print(f"  training:   {len(training_seqs)} sequences")
print(f"  baseline:   {len(baseline_seqs)} valid 80-bp sequences")
print(f"  fine-tuned: {len(finetuned_seqs)} valid 80-bp sequences")

# Three-panel motif logo comparison.
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
plot_logo(training_seqs,  f"Training set (n={len(training_seqs)})",            axes[0])
plot_logo(baseline_seqs,  f"Pretrained Evo2 + <BOS> (n={len(baseline_seqs)})", axes[1])
plot_logo(finetuned_seqs, f"Fine-tuned + <BOS> (n={len(finetuned_seqs)})",     axes[2])
axes[2].set_xlabel("position (0 = -60 from TSS, 60 = TSS)")
plt.tight_layout(); plt.show()

In [ ]:
# Motif scanning: -10 box (TATAAT) and -35 box (TTGACA), simple PWM scoring.
# These are coarse consensus PWMs from RegulonDB / standard textbook values.
MINUS_10_PWM = pd.DataFrame({
    "A": [0.04, 0.81, 0.05, 0.62, 0.55, 0.04],
    "C": [0.10, 0.06, 0.10, 0.13, 0.20, 0.05],
    "G": [0.10, 0.06, 0.10, 0.13, 0.10, 0.05],
    "T": [0.76, 0.07, 0.75, 0.12, 0.15, 0.86],
})  # TATAAT
MINUS_35_PWM = pd.DataFrame({
    "A": [0.05, 0.05, 0.05, 0.62, 0.62, 0.04],
    "C": [0.05, 0.05, 0.05, 0.13, 0.13, 0.86],
    "G": [0.05, 0.05, 0.86, 0.13, 0.13, 0.05],
    "T": [0.85, 0.85, 0.04, 0.12, 0.12, 0.05],
})  # TTGACA

def pwm_log_odds(pwm):
    bg = 0.25
    return np.log2(pwm.values / bg + 1e-6)

def best_score(seq, log_odds):
    L = log_odds.shape[0]
    if len(seq) < L: return -np.inf, -1
    arr = np.array(["ACGT".index(b) for b in seq])
    best, pos = -np.inf, -1
    for i in range(len(seq) - L + 1):
        score = sum(log_odds[k, arr[i + k]] for k in range(L))
        if score > best: best, pos = score, i
    return best, pos

minus10_lo = pwm_log_odds(MINUS_10_PWM)
minus35_lo = pwm_log_odds(MINUS_35_PWM)

def scan_population(seqs, name):
    if not seqs:
        print(f"  {name}: no valid sequences to scan")
        return np.array([]), np.array([])
    s10, s35, spacer_ok = [], [], 0
    for s in seqs:
        b10, p10 = best_score(s, minus10_lo)
        b35, p35 = best_score(s, minus35_lo)
        s10.append(b10); s35.append(b35)
        # σ70 spacer: gap between END of -35 hexamer (p35 + 6) and START of -10 hexamer (p10).
        # Canonical 17 ± 1 bp; widen to 14–20 to be lenient against PWM mis-localizations.
        spacer = p10 - (p35 + 6)
        if 14 <= spacer <= 20:
            spacer_ok += 1
    n = len(seqs)
    print(f"  {name}: median -10 score={np.median(s10):.2f}, -35 score={np.median(s35):.2f}, "
          f"valid spacer (14-20 bp)={spacer_ok}/{n} ({100 * spacer_ok / n:.1f}%)")
    return np.array(s10), np.array(s35)

print("PWM scan results (higher = stronger motif match):")
tr_10, tr_35 = scan_population(training_seqs,  "training")
bl_10, bl_35 = scan_population(baseline_seqs,  "baseline")
ft_10, ft_35 = scan_population(finetuned_seqs, "fine-tuned")

# Histogram only populations that produced valid hits, so empty baselines do not break plotting.
def stack_for_hist(arrays_with_labels):
    return [(a, lab) for a, lab in arrays_with_labels if a.size > 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, populations, title in [
    (axes[0], stack_for_hist([(tr_10, "training"), (bl_10, "baseline"), (ft_10, "fine-tuned")]),
     "-10 box (TATAAT) PWM score"),
    (axes[1], stack_for_hist([(tr_35, "training"), (bl_35, "baseline"), (ft_35, "fine-tuned")]),
     "-35 box (TTGACA) PWM score"),
]:
    if not populations:
        ax.text(0.5, 0.5, "no valid samples", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title); continue
    ax.hist([a for a, _ in populations], bins=20, label=[lab for _, lab in populations],
            stacked=False, alpha=0.6)
    ax.set_xlabel("log-odds score"); ax.set_ylabel("count"); ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# GC distribution comparison.
fig, ax = plt.subplots(figsize=(7, 4))
for seqs, label in [(training_seqs, "training"), (baseline_seqs, "baseline"), (finetuned_seqs, "fine-tuned")]:
    ax.hist([gc(s) for s in seqs], bins=25, alpha=0.5, label=f"{label} (n={len(seqs)})")
ax.set_xlabel("GC fraction"); ax.set_ylabel("count"); ax.legend()
ax.set_title("GC content distribution")
plt.tight_layout(); plt.show()

### Discussion and next steps

What to look for in the plots:
- **Logo panel.** Training set shows clear `-10` and `-35` peaks. Pretrained-with-`<BOS>` baseline should be flat (no learned signal at the BOS row); it may also yield "no valid samples" if the untrained BOS row produces non-ACGT noise — that's an expected outcome and shows up as a placeholder panel. Fine-tuned should reproduce both peaks at the right positions.
- **PWM scan.** Median `-10` and `-35` scores for the fine-tuned population should approach the training-set medians. The strictest single-number diagnostic is the **valid-spacer fraction** — the proportion of sequences where the gap between the end of the best `-35` hit and the start of the best `-10` hit lands in 14–20 bp (canonical σ70 is 17 ± 1).
- **GC content.** Fine-tuned distribution should overlap with training. If it collapses to a single value or drifts, the BOS row is producing a degenerate distribution.

Possible follow-ups:
1. **Add LoRA on early attention blocks.** If the single trained vector underfits, add `r=4` LoRA on the first ~4 attention blocks. Compare effective parameter count vs. motif quality — quantifies how much of the conditioning a single vector can carry.
2. **Strength-conditional generation.** Replace `<BOS>` with two trainable tokens — `<STRONG_PROMOTER>` and `<WEAK_PROMOTER>` — using the Urtecho et al. MPRA dataset. Tests whether multiple narrow distributions can coexist as separate trainable rows.
3. **Try the actual unconditional case.** Now that the trainable-token mechanism is validated on a concrete narrow distribution, repeat with a representative, well-mixed pretraining-style data subset to learn an unconditional `<BOS>` for whole-genome sampling — the original motivation for this pipeline.
4. **Quantitative novelty check.** BLAST fine-tuned generations against the training set; report identity distribution. If everything memorizes (>95% identity), the BOS row has become an index into training examples rather than a learned distribution.